# SSS Debris Detection: Model Comparison

Compare three YOLOv8 variants for side-scan sonar marine debris detection:

| Model | Params | Size | Key Features |
|-------|--------|------|--------------|
| **YOLOv8n** | 3.01M | ~6MB | Baseline standard YOLOv8 |
| **SS-YOLO** | 1.66M | ~3.5MB | GhostConv + FastC2f (47% lighter) |
| **YOLOv8-ESI** | 3.18M | ~6.3MB | SE attention after each C2f |

**All models are well under the 80MB deployment limit!**

In [ ]:
# Setup: clone repo and install dependencies
!git clone https://github.com/YOUR_REPO/sonar-vision.git
%cd sonar-vision
!pip install ultralytics pandas matplotlib -q

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Upload dataset
from google.colab import files
uploaded = files.upload()  # Upload e4.zip
!unzip -q e4.zip -d /content/

# Verify
import os
for split in ['train', 'val', 'test']:
    n = len([f for f in os.listdir(f'/content/e4/images/{split}') if f.endswith('.png')])
    print(f'{split}: {n} images')

## Run Comparison Training

This trains all three models and generates a comparison report.

In [ ]:
# Run the comparison script
!python scripts/train_sss_comparison.py \
    --dataset /content/e4/data.yaml \
    --epochs 100 \
    --batch 16 \
    --imgsz 512 \
    --output /content/sss_comparison \
    --models yolov8n ss_yolo yolov8_esi

## View Results

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Show comparison plot
display(Image('/content/sss_comparison/comparison.png'))

# Show CSV results
df = pd.read_csv('/content/sss_comparison/comparison.csv')
print(df[['model', 'n_params', 'mAP50', 'f1', 'precision', 'recall']].to_string(index=False))

In [ ]:
# Download best model weights
from google.colab import files

# Find best model
best = df.loc[df['f1'].idxmax()]
model_name = best['model'].replace(' ', '_').lower()
print(f"Best model: {best['model']} (F1: {best['f1']:.4f})")

# Download
files.download(f'/content/sss_comparison/{model_name}/weights/best.pt')
files.download(f'/content/sss_comparison/{model_name}/weights/best.onnx')